# Music Player Application with Kafka and Spark Streaming

## Overview
In this assignment, you will develop a music player application that streams user interaction events (like song plays, likes, and dislikes) to Apache Kafka. These events will be processed in real-time using Spark Structured Streaming to generate music recommendations based on user preferences, song metadata, and business goals to maximize revenue.

## Objectives
- Use the existing music player interface made Streamlit that randomly selects tracks.
- Stream user events to a Kafka topic.
- Process and analyze streaming data with Spark Structured Streaming.
- Implement a recommendation system based on user interactions and song metadata.
- Ensure recommendations are triggered after at least 10 user actions.

## Teams
You may work in groups of two or three. 

## Requirements

### Part 1: Music Player Interface (Streamlit)
- Modify the provided Streamlit application to include features for playing music, liking, disliking, and skipping tracks.
- For each user action, generate an event containing:
  - User ID
  - Track ID
  - Action type (Play, Like, Dislike,..)
  - Timestamp
  - If you want, you can also stream additional metadata (e.g., track length, album, artist, genre)
- Stream these events to a custom kafka topic of your choice. 
- ONE Topic per Group.

In [1]:
# Solution implemented in streamlit_app.py

### Part 2: Real-Time Data Processing with Spark, Faust, or native Kafka API

- Develop a Spark Structured Streaming application to consume events from the Kafka topic. 
- (You can also process the events using Python Kafka Api or Faust )
- Process and analyze the streaming data to track user preferences and song popularity.
- Implement logic to trigger song recommendations based on:
  - At least 10 user interactions.
  - Preferences for artists, genres, and albums,....
  - Maximization of revenue (considering factors like track pricing).


#### Setup: path, group topic name and Kafka config shared with 'streamlit_app.py'

In [2]:
import json
import os
import time
from pathlib import Path

# Spark launches a local JVM gateway that resolves the machine hostname; on some sandboxes/containers that fails, so we pin it to localhost. Harmless if not needed.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

import utils.ccloud_lib as ccloud_lib

GROUP_NAME = "group_ghanem-bilalic"
KAFKA_TOPIC = "group_ghanem-bilalic-music"  # must match KAFKA_TOPIC in streamlit_app.py
CONFIG_FILE = "kafka.config"
RECOMMENDATIONS_FILE = Path("recommendations.json")  # must match streamlit_app.py
RECOMMENDATION_THRESHOLD = 10  # must match RECOMMENDATION_THRESHOLD in streamlit_app.py

kafka_config = ccloud_lib.read_ccloud_config(CONFIG_FILE)
ccloud_lib.create_topic(kafka_config, KAFKA_TOPIC)
kafka_config

Topic group_ghanem-bilalic-music already exists


{'bootstrap.servers': '46.225.20.89:9092',
 'client.id': 'fhtw-kafka-client',
 'acks': 'all',
 'session.timeout.ms': '10000',
 'heartbeat.interval.ms': '3000'}

#### Load the track cataloge (id, name, artist, album, genre, price) from PostgreSQL

In [3]:
import psycopg2

PG_HOST = "fhtw-big-data.postgres.database.azure.com"
PG_DATABASE = "music_store"
PG_USER = "student"
PG_PASSWORD = "reRZ2pjg1WxqlwjU"

pg_conn = psycopg2.connect(host=PG_HOST, dbname=PG_DATABASE, user=PG_USER, password=PG_PASSWORD)
with pg_conn.cursor() as cur:
    cur.execute(
        """
        SELECT t.id, t.name, ar.name, al.title, g.name, t.unit_price
        FROM public.tracks t
        JOIN public.albums al ON t.album_id = al.id
        JOIN public.artists ar ON al.artist_id = ar.id
        JOIN public.genres g ON t.genre_id = g.id
        """
    )
    rows = cur.fetchall()

catalog = [
    {
        "track_id": r[0],
        "track_name": r[1],
        "artist": r[2],
        "album": r[3],
        "genre": r[4],
        "unit_price": float(r[5]),
    }
    for r in rows
]
print(f"Loaded {len(catalog)} tracks into the in-memory catalog.")

Loaded 3503 tracks into the in-memory catalog.


##### Spark session + reading & parsing the Kafka stream

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, window
from pyspark.sql.functions import count as spark_count
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

spark = (
    SparkSession.builder
    .appName("group_ghanem-bilalic-music-recommendations")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

event_schema = StructType([
    StructField("user_id", StringType()),
    StructField("track_id", IntegerType()),
    StructField("action", StringType()),
    StructField("ts", DoubleType()),
    StructField("track_name", StringType()),
    StructField("artist", StringType()),
    StructField("album", StringType()),
    StructField("genre", StringType()),
    StructField("unit_price", DoubleType()),
])

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_config["bootstrap.servers"])
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
events = (
    raw.selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), event_schema).alias("d"))
    .select("d.*")
    .withColumn("event_time", from_unixtime(col("ts")).cast("timestamp"))
)

:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ba5dcd76-c385-4df1-a306-28943caa794f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1108ms :: artifacts dl 24ms
	::

##### Song popularity (windowed aggregation)

In [5]:
song_popularity = (
    events
    .withWatermark("event_time", "2 minutes")
    .groupBy(window("event_time", "2 minutes"), "track_id", "track_name", "artist")
    .agg(spark_count("*").alias("play_count"))
)

popularity_query = (
    song_popularity.writeStream
    .outputMode("update")
    .format("memory")
    .queryName("song_popularity")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Popularity query started:", popularity_query.id)

26/07/05 22:09:13 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-12ed1dc3-eb72-4726-86b9-4d8a67113f93. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 22:09:13 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Popularity query started: 078a359c-9add-4f5c-a96b-8fe2c830a369


##### Per-user preferences and the 10-interaction trigger

In [6]:
ACTION_WEIGHTS = {"play": 1.0, "like": 3.0, "dislike": -3.0, "skip": -1.0}

user_state = {}  # user_id -> dict(genre_score, artist_score, album_score, count, seen, disliked)


def update_user_state(rows):
    """Update per-user preference counters from one micro-batch of events.

    Returns the set of user_ids whose interaction count is >= RECOMMENDATION_THRESHOLD
    after this batch, i.e. the users for whom we should (re-)generate recommendations.
    """
    triggered_users = set()
    for row in rows:
        state = user_state.setdefault(
            row.user_id,
            {"genre_score": {}, "artist_score": {}, "album_score": {}, "count": 0, "seen": set(), "disliked": set()},
        )
        weight = ACTION_WEIGHTS.get(row.action, 0.0)
        state["genre_score"][row.genre] = state["genre_score"].get(row.genre, 0.0) + weight
        state["artist_score"][row.artist] = state["artist_score"].get(row.artist, 0.0) + weight
        state["album_score"][row.album] = state["album_score"].get(row.album, 0.0) + weight
        state["count"] += 1
        state["seen"].add(row.track_id)
        if row.action == "dislike":
            state["disliked"].add(row.track_id)

        print(f"user={row.user_id} interactions={state['count']}")
        if state["count"] >= RECOMMENDATION_THRESHOLD:
            triggered_users.add(row.user_id)

    return triggered_users

### Part 3: Generating Recommendations

- Use the processed data to generate song recommendations.
- Recommendations should be dynamic, reflecting the user's recent actions and general preferences.
- Display these recommendations in the Streamlit app, however you prefer.
- Custom table from a database is also fine.

In [7]:
ARTIST_WEIGHT, GENRE_WEIGHT, ALBUM_WEIGHT, REVENUE_WEIGHT = 1.5, 1.0, 1.0, 2.0
TOP_N = 5

if not RECOMMENDATIONS_FILE.exists():
    RECOMMENDATIONS_FILE.write_text("{}")


def write_recommendations_atomic(all_recs):
    """Write recommendations.json without ever leaving a half-written file behind."""
    tmp = RECOMMENDATIONS_FILE.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(all_recs, indent=2))
    os.replace(tmp, RECOMMENDATIONS_FILE)


def recommend_for_user(state, top_n=TOP_N):
    """Score the catalog for one user's state and return the top_n (score, track) pairs."""
    genre_score, artist_score, album_score = state["genre_score"], state["artist_score"], state["album_score"]
    seen, disliked = state["seen"], state["disliked"]

    def score(track):
        return (
            genre_score.get(track["genre"], 0.0) * GENRE_WEIGHT
            + artist_score.get(track["artist"], 0.0) * ARTIST_WEIGHT
            + album_score.get(track["album"], 0.0) * ALBUM_WEIGHT
            + track["unit_price"] * REVENUE_WEIGHT
        )

    # Prefer tracks the user hasn't already interacted with; fall back to the full
    # (non-disliked) catalog only if that pool is too small.
    candidates = [t for t in catalog if t["track_id"] not in disliked and t["track_id"] not in seen]
    if len(candidates) < top_n:
        candidates = [t for t in catalog if t["track_id"] not in disliked]

    scored = sorted(((score(t), t) for t in candidates), key=lambda x: x[0], reverse=True)
    return scored[:top_n]


def generate_and_store_recommendations(user_id):
    """Score, rank and persist the top_n recommendations for one user."""
    state = user_state[user_id]
    top_tracks = recommend_for_user(state)

    try:
        all_recs = json.loads(RECOMMENDATIONS_FILE.read_text())
    except json.JSONDecodeError:
        all_recs = {}

    all_recs[user_id] = {
        "interactions": state["count"],
        "generated_at": time.time(),
        "tracks": [
            {
                "rank": rank,
                "track_id": track["track_id"],
                "track_name": track["track_name"],
                "artist": track["artist"],
                "album": track["album"],
                "genre": track["genre"],
                "unit_price": track["unit_price"],
                "score": round(track_score, 3),
            }
            for rank, (track_score, track) in enumerate(top_tracks, start=1)
        ],
    }
    write_recommendations_atomic(all_recs)
    print(f"  -> wrote {len(top_tracks)} recommendations for {user_id}")


def process_batch(batch_df, batch_id):
    """foreachBatch entry point: update state (Part 2), then generate recommendations
    for every user who has reached the threshold (Part 3)."""
    rows = batch_df.collect()
    if not rows:
        return
    print(f"[batch {batch_id}] {len(rows)} events")
    triggered_users = update_user_state(rows)
    for user_id in triggered_users:
        generate_and_store_recommendations(user_id)

In [ ]:
recommendation_query = (
    events.writeStream
    .foreachBatch(process_batch)
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Recommendation query started:", recommendation_query.id)

26/07/05 22:09:15 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-547536f6-79ec-453e-a1e5-8fd1ab0020c9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 22:09:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Recommendation query started: 72a5eeaf-4c89-42e0-b46e-1befdf21a912


26/07/05 22:09:15 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/07/05 22:09:15 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


[batch 0] 48 events
user=nb-exec-test-user-2 interactions=1
user=nb-exec-test-user-2 interactions=2
user=nb-exec-test-user-2 interactions=3
user=nb-exec-test-user-2 interactions=4
user=nb-exec-test-user-2 interactions=5
user=nb-exec-test-user-2 interactions=6
user=nb-exec-test-user-2 interactions=7
user=nb-exec-test-user-2 interactions=8
user=nb-exec-test-user-2 interactions=9
user=nb-exec-test-user-2 interactions=10
user=nb-exec-test-user-2 interactions=11
user=nb-exec-test-user-2 interactions=12
user=user-d940b497 interactions=1
user=user-d940b497 interactions=2
user=user-d940b497 interactions=3
user=user-d940b497 interactions=4
user=user-d940b497 interactions=5
user=user-d940b497 interactions=6
user=user-d940b497 interactions=7
user=user-d940b497 interactions=8
user=user-d940b497 interactions=9
user=user-d940b497 interactions=10
user=user-d940b497 interactions=11
user=user-452aff53 interactions=1
user=user-452aff53 interactions=2
user=user-452aff53 interactions=3
user=user-452aff53 

26/07/05 22:09:25 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 10149 milliseconds
26/07/05 22:10:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 98841 milliseconds
26/07/05 22:12:08 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 75449 milliseconds


...

[batch 12] 2 events
user=user-1b54fa7b interactions=3
user=user-1b54fa7b interactions=4


26/07/05 22:25:59 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 59544 milliseconds


[batch 13] 10 events
user=user-1b54fa7b interactions=5
user=user-1b54fa7b interactions=6
user=user-1b54fa7b interactions=7
user=user-1b54fa7b interactions=8
user=user-1b54fa7b interactions=9
user=user-1b54fa7b interactions=10
user=user-1b54fa7b interactions=11
user=user-1b54fa7b interactions=12
user=user-1b54fa7b interactions=13
user=user-1b54fa7b interactions=14
  -> wrote 5 recommendations for user-1b54fa7b


[batch 14] 1 events
user=user-1b54fa7b interactions=15
  -> wrote 5 recommendations for user-1b54fa7b


26/07/05 22:27:12 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 62707 milliseconds
26/07/05 22:27:12 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 73944 milliseconds
26/07/05 22:28:29 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 76785 milliseconds
26/07/05 22:29:43 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 73736 milliseconds
